<a href="https://colab.research.google.com/github/JakeFRCSE/geometry-of-truth-reproduction/blob/main/notebooks/localization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0. Loading Data

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import pandas as pd
from torch.utils.data import DataLoader, Dataset

In [ ]:
BASE_DIR = Path("/content/drive/MyDrive/research/research-cycle1/throw-away-projects/geometry-of-truth")
DATASET_DIR = BASE_DIR / "dataset"
dataset_names = ["cities.csv", "neg_cities.csv"]
patching_prompts = {
    "cities": {
        "false_prompt" : "The city of Tokyo is in Japan. This statement is: TRUE\nThe city of Hanoi is in Poland. This statement is: FALSE\nThe city of Chicago is in Canada. This statement is:",
        "true_prompt" : "The city of Tokyo is in Japan. This statement is: TRUE\nThe city of Hanoi is in Poland. This statement is: FALSE\nThe city of Toronto is in Canada. This statement is:",
    },
}

In [ ]:
class DataFrameDataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        statement = row["statement"]
        label = row["label"]
        return {"statement": statement,
                "label": label}

def load_dataset(dataset_name: str) -> pd.DataFrame:
    dataset_path = DATASET_DIR / dataset_name
    return pd.read_csv(dataset_path)

In [ ]:
datasets = { dataset_name.split(".")[0] : load_dataset(dataset_name) for dataset_name in dataset_names}
datasets_torch = {dataset_name : DataFrameDataset(dataset) for dataset_name, dataset in datasets.items()}
dataloaders = { dataset_name: DataLoader(dataset, batch_size=32, shuffle=False) for dataset_name, dataset in datasets_torch.items()}

cities = dataloaders["cities"]
neg_cities = dataloaders["neg_cities"]

# 1. Loading Model

In [ ]:
!pip install transformer_lens

In [ ]:
from transformer_lens import HookedTransformer, utils, ActivationCache
import torch
from tqdm.auto import tqdm

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
category = "cities"
model_name = "EleutherAI/pythia-6.9b"

model = HookedTransformer.from_pretrained(model_name,
                                          dtype=torch.bfloat16,
                                          device=device)

# 2. Defining Functions

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def tokenize_patching_prompt(model: HookedTransformer, patching_category: str) -> dict:
    patching_pair = patching_prompts[patching_category]
    tokenized_patching_pair = {true_false: model.to_tokens(prompt).squeeze(0) for true_false, prompt in patching_pair.items()}
    assert len(set(len(tokens) for tokens in tokenized_patching_pair.values())) == 1, \
    "True/False prompts must have the same token length"
    return tokenized_patching_pair

def find_change_token_idx(prompt_pair: dict) -> int:
    true_prompt = prompt_pair["true_prompt"]
    false_prompt = prompt_pair["false_prompt"]
    diff_idx = (true_prompt != false_prompt).nonzero(as_tuple=True)[0]
    return diff_idx.item()

def cache_true_activation(model: HookedTransformer, true_prompt_tokens: torch.tensor, change_token_idx: int) -> torch.tensor:
    true_activation = []

    _, cache = model.run_with_cache(true_prompt_tokens,
                                    return_type=None,
                                    return_cache_object=True)

    for l in range(model.cfg.n_layers):
        true_activation.append(cache["resid_post", l][:,change_token_idx:])

    return torch.cat(true_activation, dim=0)

@torch.no_grad()
def compute_intervention_scores(
    model: HookedTransformer,
    true_activations: torch.Tensor,
    false_prompt_tokens: torch.Tensor,
    change_token_idx: int,
) -> torch.Tensor:
    TRUE_TOKEN = model.to_single_token("TRUE")
    FALSE_TOKEN = model.to_single_token("FALSE")

    n_layers, n_tokens, d_model = true_activations.shape
    scores = torch.zeros((n_layers, n_tokens), device=model.cfg.device)

    for l in tqdm(range(n_layers)):
        for i in range(n_tokens):
            patched_vector = true_activations[l, i, :]
            patch_pos = change_token_idx + i

            def fwd_hook_fn(activation, hook, patched_vector=patched_vector, patch_pos=patch_pos):
                activation[:, patch_pos, :] = patched_vector
                return activation

            hook_point = utils.get_act_name("resid_post", l)

            logits = model.run_with_hooks(
                false_prompt_tokens,
                fwd_hooks=[(hook_point, fwd_hook_fn)],
                return_type="logits"
            )

            log_probs = logits[0, -1, :].log_softmax(dim=-1)
            p_true = log_probs[TRUE_TOKEN]
            p_false = log_probs[FALSE_TOKEN]

            scores[l, i] = p_true - p_false

    return scores

def compute_localized_scores(model: HookedTransformer, category: str) -> torch.tensor:

    tokenized_patching_prompt = tokenize_patching_prompt(model, category)
    change_token_idx = find_change_token_idx(tokenized_patching_prompt)

    true_activations = cache_true_activation(model, tokenized_patching_prompt["true_prompt"], change_token_idx)
    intervention_scores = compute_intervention_scores(model, true_activations, tokenized_patching_prompt["false_prompt"], change_token_idx)

    return intervention_scores

def save_intervention_heatmap(
    intervention_scores: torch.Tensor,
    model: HookedTransformer,
    category: str,
    results_dir: Path,
    cmap: str = "Blues",
    figsize: tuple[int, int] = (12, 8),
    filename: str | None = None,
) -> None:
    scores = intervention_scores.detach().cpu()

    tokenized_patching_prompt = tokenize_patching_prompt(model, category)
    tokens = model.to_str_tokens(tokenized_patching_prompt["false_prompt"])

    change_token_idx = find_change_token_idx(tokenized_patching_prompt)
    tokens = tokens[change_token_idx:]

    if filename is None:
        filename = f"{category}_intervention_heatmap.png"

    results_dir.mkdir(parents=True, exist_ok=True)
    file_path = results_dir / filename

    plt.figure(figsize=figsize)

    sns.heatmap(
        scores,
        cmap=cmap,
        vmin=scores.min().item(),
        vmax=scores.max().item(),
        cbar_kws={"label": "Log(P_True) - Log(P_False)"},
    )

    plt.xlabel("Token")
    plt.ylabel("Layer")
    plt.title(f"Intervention Scores Heatmap | {model_name} | {category}")
    plt.xticks(
        ticks=range(len(tokens)),
        labels=tokens,
        rotation=45,
        ha="right",
    )

    plt.tight_layout()
    plt.savefig(file_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

# 3. Run & Visualization

In [ ]:
SAVE_DIR = BASE_DIR / "results" / model_name / category
SAVE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
intervention_scores = compute_localized_scores(model, category)

save_intervention_heatmap(
    intervention_scores=intervention_scores,
    model=model,
    category=category,
    results_dir=SAVE_DIR,
)